In [1]:
%matplotlib inline
import openmc
import openmc.deplete
import matplotlib.pyplot as plt
import glob
import shutil
import os
import zipfile
import sys
import openmc.examples
import numpy as np
import matplotlib.pyplot as plt
import math
import psutil
import time
print('Done')

Done


In [2]:
openmc.config['cross_sections'] = '/home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/cross_sections.xml'
output_dir = "/home/kokomoor/reactor_phys_final_proj/Problem7"
process = psutil.Process()
mem_before = process.memory_info().rss / 1024**2  # in MB
start_time = time.time()

In [3]:
fuel = openmc.Material(name='fuel')
fuel.add_nuclide('U238', 0.7502, 'wo')
fuel.add_nuclide('U235', 0.1376, 'wo')
fuel.add_nuclide('O16', 0.0897, 'wo')
fuel.add_nuclide('C0', 0.0224, 'wo')
fuel.set_density('g/cc', 10.5)

buff = openmc.Material(name='Buffer')
buff.add_nuclide('C0', 1.0, 'wo')
buff.set_density('g/cm3', 1.0)

iPyC = openmc.Material(name='iPyC')
iPyC.add_nuclide('C0', 1.0, 'wo')
iPyC.set_density('g/cm3', 1.9)

oPyC = openmc.Material(name='oPyC')
oPyC.add_nuclide('C0', 1.0, 'wo')
oPyC.set_density('g/cm3', 1.9)

SiC = openmc.Material(name='SiC')
SiC.add_nuclide('C0', 0.5, 'wo')
SiC.add_element('Si', 0.5, 'wo')
SiC.set_density('g/cm3', 3.2)

coolant = openmc.Material(name='coolant')
#NaFZrF4
coolant.add_nuclide('F19',0.4537, 'wo')
coolant.add_nuclide('Na23',0.1475, 'wo')
coolant.add_nuclide('Zr91',0.3988, 'wo')
coolant.set_density('g/cm3',2.96)

graphite = openmc.Material(name='moderator')
graphite.add_nuclide('C0',  0.99998985, 'wo')
graphite.add_nuclide('B10', 0.00000015, 'wo')
graphite.add_nuclide('N14', 0.00001, 'wo')
graphite.set_density('g/cc', 1.65)

In [4]:
# fuel_density = 10.5
# buffer_density = 1.0
# iPyC_density = 1.9
# SiC_density = 3.2
# oPyC_density = 1.9
# graphite_density = 1.65

r_fuel = 0.02125
r_buffer = 0.03125
r_iPyC = 0.03475
r_SiC = 0.03835
r_oPyC = 0.04225
r_TRISO = 0.04225

V_fuel   = 4/3 * math.pi * r_fuel**3
V_buffer = 4/3 * math.pi * (r_buffer**3 - r_fuel**3)
V_iPyC   = 4/3 * math.pi * (r_iPyC**3 - r_buffer**3)
V_SiC    = 4/3 * math.pi * (r_SiC**3 - r_iPyC**3)
V_oPyC   = 4/3 * math.pi * (r_oPyC**3 - r_SiC**3)
# 0.0003159139103058562
V_TRISO  = 4/3 * math.pi * r_oPyC**3

assert (V_fuel + V_buffer + V_iPyC + V_SiC + V_oPyC) == V_TRISO, "Volume fractions should sum to total volume of TRISO"
#VOL_COMPACT = 2.53

# This is the fraction of a single TRISO pellet that each
# component material represents
# As of now, fuel stands for the uranium in the core of the TRISO
# Itself, not the entire TRISO
fuel_frac = V_fuel / V_TRISO
buffer_frac = V_buffer / V_TRISO
iPyC_frac = V_iPyC / V_TRISO
SiC_frac = V_SiC / V_TRISO
oPyC_frac = V_oPyC / V_TRISO

In [ ]:
# defines the radius of  the fuel compact
# outer_radius_triso = 422.5*1e-4
# num_trisos = openmc.model.pack_spheres(radius=outer_radius_triso, region=region, pf=0.35)
# print("FUELNUM", len(num_trisos))
#########################################
# Calculated from the above commented code, essentially we see how many TRISO spheres we can fit in the original
# Fuel compact cylinder at the original packing fraction, which comes out to 2804 spheres. Ergo we're using 2804 spheres in
# the heterogeneous model and we need to use the equivalent amount of material, so we multiply V_TRISO by 2804 to
# arrive at the total volume of TRISO material in the compact.
#########################################
# Annoyingly, this represents volume of all triso material, not volume of just fuel material
HELD_CONSTANT_V_FUEL = 0.8858226044976208

# Here we set the compact radii we want to simulate and simulate them in a loop
r_compacts = [0.635]

# Calculate Border between fuel and graphite regions for a heterogeneous fuel compact of homogenized triso material
# surrounded by graphite
# TRISO_COMPACT_RADIUS = math.sqrt(HELD_CONSTANT_V_FUEL/(2*math.pi))

for r_compact in r_compacts:
    # RRPT Geometry Definitions    
    # Define radii for concentric rings (chosen to preserve 35% fuel and 65% graphite by area)
    
    inner_fuel_allocation = 0.01  # user-adjustable (0.5 = 50/50 split)
    original_pf = 0.35           # total packing fraction (TRISO volume fraction)
    inner_fuel_frac_RRPT = original_pf * inner_fuel_allocation
    outer_fuel_frac_RRPT = original_pf * (1.0 - inner_fuel_allocation)
    inner_graphite_frac = (1.0 - original_pf)*0.01
    middle_graphite_frac = (1.0 - original_pf)*0.98
    outer_graphite_frac = (1.0 - original_pf)*0.01
    # Verify the fractions sum to 1.0 (100% of compact volume)
    assert abs(inner_fuel_frac_RRPT + outer_fuel_frac_RRPT + inner_graphite_frac + middle_graphite_frac + outer_graphite_frac - 1.0) < 1e-9

    # Using cumulative fractions to find radii
    r_inner_graphite  = math.sqrt(inner_graphite_frac) * r_compact
    r_inner_fuel      = math.sqrt(inner_graphite_frac + inner_fuel_frac_RRPT) * r_compact
    r_middle_graphite = math.sqrt(inner_graphite_frac + inner_fuel_frac_RRPT + middle_graphite_frac) * r_compact
    r_outer_fuel      = math.sqrt(inner_graphite_frac + inner_fuel_frac_RRPT + middle_graphite_frac + outer_fuel_frac_RRPT) * r_compact
    r_outer_graphite  = r_compact

    print(f"RING RADII: Inner Graphite {r_inner_graphite} Inner Fuel {r_inner_fuel} Middle Graphite {r_middle_graphite} Outer Fuel {r_outer_fuel} Outer Graphite {r_outer_graphite}")
    
    # Create radial surfaces (cylinders centered on fuel compact)
    inner_graphite_cyl  = openmc.ZCylinder(r=r_inner_graphite)
    inner_fuel_cyl      = openmc.ZCylinder(r=r_inner_fuel)
    middle_graphite_cyl = openmc.ZCylinder(r=r_middle_graphite)
    outer_fuel_cyl      = openmc.ZCylinder(r=r_outer_fuel)
    outer_boundary_cyl  = openmc.ZCylinder(r=r_compact)  # compact outer surface (may already exist)

    # Geometry definitions
    # We define a slightly smaller compact radius to avoid situations
    # where TRISOs would be cut by the boundary.
    fuel_radius = openmc.ZCylinder(r=r_compact) 
    coolant_radius = openmc.ZCylinder(r=0.35) 
    compact_radius = openmc.ZCylinder(r=r_compact) 
    top_compact = openmc.ZPlane(z0=+1.0, surface_id=600)
    bottom_compact = openmc.ZPlane(z0=-1.0, surface_id=500) 
    region = -compact_radius & -top_compact & +bottom_compact
    
    # Volume of the fuel compact based on radius
    total_volume_compact = math.pi*(r_compact**2)*2

    # calculate the new packing fraction by dividing the
    # constant fuel volume by the new volume of the fuel compact
    new_pacfrac = HELD_CONSTANT_V_FUEL / total_volume_compact
    
    #We have the fraction of each fuel in the TRISO, now we need its fraction within the fuel compact
    compact_fuel_frac = fuel_frac * new_pacfrac
    compact_buffer_frac = buffer_frac * new_pacfrac
    compact_iPyC_frac = iPyC_frac * new_pacfrac
    compact_SiC_frac = SiC_frac * new_pacfrac
    compact_oPyC_frac = oPyC_frac * new_pacfrac
    compact_graphite_frac = 1 - new_pacfrac # 1 - packing fraction
    print()
    print("This should be 1: ", compact_fuel_frac + compact_buffer_frac + compact_iPyC_frac + compact_SiC_frac + compact_oPyC_frac + compact_graphite_frac)
    
    materials = [fuel, buff, iPyC, SiC, oPyC, graphite]
    fractions = [compact_fuel_frac, compact_buffer_frac, compact_iPyC_frac, compact_SiC_frac, compact_oPyC_frac, compact_graphite_frac]
    
    homog_nuclides = {}  # dictionary to accumulate homogenized densities
    for mat, frac in zip(materials, fractions):
        comp = mat.get_nuclide_atom_densities()  # dict of {nuclide: (density, units)}
        for nuclide, dens in comp.items():
            homog_nuclides[nuclide] = homog_nuclides.get(nuclide, 0.0) + dens * frac

    RRPT_materials = [fuel, buff, iPyC, SiC, oPyC]
    RRPT_fractions = [fuel_frac, buffer_frac, iPyC_frac, SiC_frac, oPyC_frac]
    RRPT_nuclides = {}  # dictionary to accumulate RRPT Fuel densities
    for mat, frac in zip(RRPT_materials, RRPT_fractions):
        comp = mat.get_nuclide_atom_densities()  # dict of {nuclide: (density, units)}
        for nuclide, dens in comp.items():
            RRPT_nuclides[nuclide] = RRPT_nuclides.get(nuclide, 0.0) + dens * frac
    
    # Build the homogenous material, and defines its volume
    # as the total volume of its compact
    homog_mat = openmc.Material(name="Homogenized Fuel Compact")
    for nuclide, dens in homog_nuclides.items():
        homog_mat.add_nuclide(nuclide, dens)  # dens is in atom/b-cm units
    homog_mat.set_density('sum')  # interpret added nuclide values as absolute densities
    homog_mat.volume = total_volume_compact

    #Build The RRPT Fuel Material
    RRPT_mat = openmc.Material(name="RRPT Fuel Compact")
    for nuclide, dens in RRPT_nuclides.items():
        RRPT_mat.add_nuclide(nuclide, dens)  # dens is in atom/b-cm units
    RRPT_mat.set_density('sum')  # interpret added nuclide values as absolute densities
    RRPT_mat.volume = HELD_CONSTANT_V_FUEL
    
    materials_file = openmc.Materials([fuel, graphite, buff, iPyC, oPyC, SiC,coolant, homog_mat, RRPT_mat])
    materials_file.export_to_xml()

    #This just defines reactor temperatures for the life of the experiment
    fuel.temperature = 1000
    buff.temperature = 900
    iPyC.temperature = 900
    oPyC.temperature = 900
    SiC.temperature = 900
    coolant.temperature = 900
    graphite.temperature = 900
    
    # Define cells for each ring region
    inner_graphite_cell = openmc.Cell(name="Inner Graphite Ring",
        fill=graphite,
        region=(-inner_graphite_cyl & +bottom_compact & -top_compact))
    
    inner_fuel_cell = openmc.Cell(name="Inner Fuel Ring",
        fill=RRPT_mat,
        region=(+inner_graphite_cyl & -inner_fuel_cyl & +bottom_compact & -top_compact))
    
    middle_graphite_cell = openmc.Cell(name="Middle Graphite Ring",
        fill=graphite,
        region=(+inner_fuel_cyl & -middle_graphite_cyl & +bottom_compact & -top_compact))
    
    outer_fuel_cell = openmc.Cell(name="Outer Fuel Ring",
        fill=RRPT_mat,
        region=(+middle_graphite_cyl & -outer_fuel_cyl & +bottom_compact & -top_compact))
    
    outer_graphite_cell = openmc.Cell(name="Outer Graphite Ring",
        fill=graphite,
        region=(+outer_fuel_cyl & -outer_boundary_cyl & +bottom_compact & -top_compact))

    # Define the hexagonal prism region for one lattice cell (using half the flat-to-flat pitch as apothem)
    hex_cell_region = openmc.model.hexagonal_prism(edge_length=1.3, orientation='x')
    
    # Define a graphite wrapper cell filling outside the outer compact cylinder up to the hex cell boundary
    wrapper_cell = openmc.Cell(name='Outer graphite wrapper',
                               fill=graphite,
                               region=+fuel_radius)
    
    # Create a universe for the fuel compact consisting of the five ring cells
    compact_universe = openmc.Universe(name="RRPT Fuel Compact", 
                                       cells=[inner_graphite_cell, inner_fuel_cell,
                                              middle_graphite_cell, outer_fuel_cell,
                                              outer_graphite_cell, wrapper_cell])

    top = openmc.ZPlane(z0=+1, boundary_type='periodic', surface_id=700)
    bottom = openmc.ZPlane(z0=-1, boundary_type='periodic', surface_id=800) 
    top.periodic_surface = bottom
    bottom.periodic_surface = top
    
    fuel_region = -fuel_radius
    moder_fuel = +fuel_radius
    coolant_region = -coolant_radius 
    moder_coolant = +coolant_radius 
     
    fuel_cell = openmc.Cell(fill=homog_mat, region=fuel_region)
    moder_fuel_cell = openmc.Cell(fill=graphite, region=moder_fuel)
    graphite_cell = openmc.Cell(fill=graphite)
    
    coolant_cell = openmc.Cell(fill=coolant, region=coolant_region)
    moder_coolant_cell = openmc.Cell(fill=graphite, region=moder_coolant)
    
    fuel_u = openmc.Universe(cells=(fuel_cell,moder_fuel_cell))
    graphite_u = openmc.Universe(cells=[graphite_cell])
    coolant_u = openmc.Universe(cells=[coolant_cell,moder_coolant_cell])
    
    inner = [coolant_u]
    outer = [coolant_u,compact_universe,coolant_u,compact_universe,coolant_u,compact_universe]
    
    #An hexagonal lattice is defined with a fuel universe surrounded by a graphite universe.
    
    hex_lat = openmc.HexLattice(name='assembly')
    hex_lat.center = (0., 0.)
    # PRIOR 1.8796
    hex_lat.pitch = (1.8796,)
    hex_lat.orientation = 'x'
    hex_lat.outer = graphite_u
    hex_lat.universes = [outer, inner]
    
    #An hexagonal bounding box is defined around the lattice.
    
    # Create the prism that will contain the lattice PRIOR WAS 2.8194
    outer_surface = openmc.model.hexagonal_prism(edge_length=2.8194, orientation='x', boundary_type='periodic')
    
    # Fill a cell with the lattice. This cell is filled with the lattice and contained within the prism.
    main_assembly = openmc.Cell(fill=hex_lat, region=outer_surface & -top & +bottom)
    
    # Create a universe that contains both 
    root = openmc.Universe(cells=[main_assembly])
    
    root.plot(origin = (0,0,0), pixels=(200, 200), width = (6.,6.), color_by = 'material')
    
    geom = openmc.Geometry(root)
    geom.export_to_xml()
    
    # OpenMC simulation parameters
    lower_left = [-2.8, -2.8, -1.0]
    upper_right = [ 2.8,  2.8,  1.0]
    # lower_left = [-3, -3, -1]
    # upper_right = [3, 3, 1]
    uniform_dist = openmc.stats.Box(lower_left, upper_right)
    src = openmc.IndependentSource(space=uniform_dist)#, constraints={'fissionable': True})
    
    settings = openmc.Settings()
    settings.source = src
    settings.batches = 100
    settings.inactive = 25
    settings.particles = 2000
    settings.temperature = {'method': 'interpolation','range':(293.15,1923.15)}
    
    settings.export_to_xml()
    
    model = openmc.examples.pwr_pin_cell()
    
    model.tallies
    
    # Create equal-lethargy energies to put in filter
    energies = np.logspace(np.log10(1e-5), np.log10(20.0e6), 501)
    e_filter = openmc.EnergyFilter(energies)
    
    model_tallies = openmc.Tallies()
    
    # Create tally with energy filter
    flux_tally = openmc.Tally(name='Flux')
    flux_tally.filters = [e_filter]
    flux_tally.scores = ['flux']
    model_tallies.append(flux_tally)
    
    # Set model tallies
    model.tallies = model_tallies
    
    openmc.mgxs.GROUP_STRUCTURES.keys()
    
    model.tallies
    print("THIS IS YOUR 10/80/10 30/70")
    openmc.run()
    
    fuel.volume = 2804*4/3*3.1416*(212.5*1e-4)**3
    
    # Create depletion "operator"
    model = openmc.Model(geometry=geom, materials=materials_file, settings=settings)
    op = openmc.deplete.CoupledOperator(model, "chain_casl_pwr.xml")
    
    # Perform simulation using the predictor algorithm
    # time_steps = [1.0, 2.0, 10.0, 20.0, 30.0]  # days
    time_steps = [1.0, 10.0, 100.0, 254.0] # days
    power = 1000  
    integrator = openmc.deplete.PredictorIntegrator(op, time_steps, power, timestep_units='d')
    integrator.integrate()
    mem_after = process.memory_info().rss / 1024**2  # in MB
    end_time = time.time()
    # # Define directory and exclusion
    # cwd = os.getcwd()
    # exclude_dir = os.path.join(cwd, 'endfb-vii.1-hdf5')
    # # Construct zip filename one level up
    # dir_name = os.path.basename(cwd.rstrip('/'))
    # zip_name = os.path.join('..', f"{dir_name}_{r_compact}.zip")
    # try:
    #     with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    #         for root, dirs, files in os.walk(cwd):
    #             # Skip the excluded directory
    #             if os.path.commonpath([exclude_dir, root]) == exclude_dir:
    #                 continue
    #             for file in files:
    #                 filepath = os.path.join(root, file)
    #                 arcname = os.path.relpath(filepath, cwd)
    #                 zipf.write(filepath, arcname)
        
    #     print(f"Zipped current directory to {zip_name}, excluding '{exclude_dir}'")
    # except Exception as e:
    #     print("Error saving zip file: ", e)

RING RADII: Inner Graphite 0.05119533670169579 Inner Fuel 0.0635 Middle Graphite 0.5107705698256312 Outer Fuel 0.63293288546259 Outer Graphite 0.635

This should be 1:  1.0


/home/kokomoor/.conda/envs/openmc-env/lib/python3.10/site-packages/openmc/model/funcs.py:124: FutureWarning: The hexagonal_prism(...) function has been replaced by the HexagonalPrism(...) class. Future versions of OpenMC will not accept hexagonal_prism.
  warn("The hexagonal_prism(...) function has been replaced by the "


THIS IS YOUR 10/80/10 30/70
                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
          

 Reading Mo92 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Mo92.h5
 Reading Mo94 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Mo94.h5
 Reading Mo95 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Mo95.h5
 Reading Mo96 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Mo96.h5
 Reading Mo97 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Mo97.h5
 Reading Mo98 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Mo98.h5
 Reading Mo99 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Mo99.h5


 Reading Mo100 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Mo100.h5
 Reading Tc99 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Tc99.h5
 Reading Ru100 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ru100.h5
 Reading Ru101 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ru101.h5
 Reading Ru102 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ru102.h5
 Reading Ru103 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ru103.h5
 Reading Ru104 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ru104.h5
 Reading Ru105 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ru105.h5
 Reading Ru106 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ru106.h5
 Reading Rh103 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Rh103.h5
 Reading Rh105 from
 /home/kokomoor/reactor_phys_final_proj/en

...

 Reading Xe135 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Xe135.h5
 Reading Xe136 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Xe136.h5
 Reading Cs133 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Cs133.h5
 Reading Cs134 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Cs134.h5
 Reading Cs135 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Cs135.h5
 Reading Cs136 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Cs136.h5
 Reading Cs137 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Cs137.h5
 Reading Ba134 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ba134.h5


          1200K
          2500K


 Reading Ba137 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ba137.h5
 Reading Ba140 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ba140.h5
 Reading La139 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/La139.h5
 Reading La140 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/La140.h5
 Reading Ce140 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ce140.h5
 Reading Ce141 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ce141.h5
 Reading Ce142 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ce142.h5
 Reading Ce143 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ce143.h5
 Reading Ce144 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Ce144.h5
 Reading Pr141 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Pr141.h5
 Reading Pr142 from
 /home/kokomoor/reactor_phys_final_proj/

          1200K
          2500K


 Reading Eu157 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Eu157.h5
 Reading Gd152 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Gd152.h5
 Reading Gd154 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Gd154.h5
 Reading Gd155 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Gd155.h5
 Reading Gd156 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Gd156.h5
 Reading Gd157 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Gd157.h5
 Reading Gd158 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Gd158.h5
 Reading Gd160 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Gd160.h5
 Reading Tb159 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Tb159.h5
 Reading Tb160 from
 /home/kokomoor/reactor_phys_final_proj/endfb-vii.1-hdf5/neutron/Tb160.h5
 Reading Dy160 from
 /home/kokomoor/reactor_phys_final_proj/